In [5]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [6]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [7]:
# check the number of words
len(words)

32033

In [8]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [9]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
    
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append (sliding window)

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [10]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [11]:
C = torch.randn((27, 2))

In [12]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [14]:
emb

tensor([[[-0.4029,  0.6378],
         [-0.4029,  0.6378],
         [-0.4029,  0.6378]],

        [[-0.4029,  0.6378],
         [-0.4029,  0.6378],
         [ 1.1123, -0.3502]],

        [[-0.4029,  0.6378],
         [ 1.1123, -0.3502],
         [-0.5468,  0.2399]],

        [[ 1.1123, -0.3502],
         [-0.5468,  0.2399],
         [-0.5468,  0.2399]],

        [[-0.5468,  0.2399],
         [-0.5468,  0.2399],
         [-0.2415,  0.8963]],

        [[-0.4029,  0.6378],
         [-0.4029,  0.6378],
         [-0.4029,  0.6378]],

        [[-0.4029,  0.6378],
         [-0.4029,  0.6378],
         [ 0.6795,  1.1817]],

        [[-0.4029,  0.6378],
         [ 0.6795,  1.1817],
         [-0.5609,  0.6237]],

        [[ 0.6795,  1.1817],
         [-0.5609,  0.6237],
         [-0.4231, -0.1591]],

        [[-0.5609,  0.6237],
         [-0.4231, -0.1591],
         [ 1.8154, -0.6729]],

        [[-0.4231, -0.1591],
         [ 1.8154, -0.6729],
         [-0.4231, -0.1591]],

        [[ 1.8154, -0

In [13]:
W1 = torch.randn((6, 100))
b1 = torch.rand(100)

In [ ]:
# emb.reshape(32, 6).shape
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1).shape
# torch.cat(torch.unbind(emb, 1), 1).shape
emb.view(32, 6).shape

torch.Size([32, 6])

In [ ]:
# remember to always check broadcasting rules for every vector

# h = emb.view(emb[0], 6) @ W1 + b1
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) 
h

tensor([[ 0.7083,  0.9565,  0.9971,  ...,  0.7814, -0.9819, -0.9049],
        [ 0.2101,  0.1803,  0.7426,  ..., -0.2068, -0.8150, -0.8513],
        [-0.4995,  0.3604,  0.9915,  ..., -0.5891, -0.8791,  0.1535],
        ...,
        [-0.8680,  0.5126,  0.9335,  ..., -0.6791,  0.3391,  0.5132],
        [ 0.9862,  0.3447, -0.8773,  ...,  0.9993,  0.8282, -0.4202],
        [ 0.8704,  0.8763,  0.0128,  ...,  0.9995, -0.3147,  0.0450]])

In [23]:
h.shape

torch.Size([32, 100])